### Developing human heart - prepare: sections, reference, genes by age, cell2location, regions (mp_heart_dvlp01)

**Paper**: Lázár E, Mauron R, Andrusivová Ž, Foyer J, He M, Larsson L, Shakari N, Salas SM, Avenel C, Sariyar S, Hansen JN, Vicari M, Czarnewski P, Braun E, Li X, Bergmann O, Sylvén C, Lundberg E, Linnarsson S, Nilsson M, Sundström E, Adameyko I, Lundeberg J. *Spatiotemporal gene expression and cellular dynamics of the developing human heart.* Nat Genet. 2025 Nov;57(11):2756-2771. doi: [10.1038/s41588-025-02352-6](https://doi.org/10.1038/s41588-025-02352-6). PMID: 41162788; PMCID: PMC12597827.

Code: [github.com/rmauron/HDCA_heart_dev](https://github.com/rmauron/HDCA_heart_dev) (Zenodo 15912657) - Data: Mendeley [part 1](https://doi.org/10.17632/fhtb99mdzd.1) (Space Ranger, Cell Ranger, metadata) and [part 2](https://doi.org/10.17632/w65jtfsvpr.1) (Seurat RDS) - [Interactive viewer](https://hdcaheart.serve.scilifelab.se/web/index.html)

The authors combined **Visium spatial transcriptomics** (38 heart sections from 16 hearts, PCW 6-12; 69,114 tissue spots), **10x Chromium scRNA-seq** (76,991 cells, PCW 5.5-14; 31 coarse-grained and 72 fine-grained cell states) and in situ sequencing (150 genes, 4 hearts) to map early human cardiogenesis: the pacemaker-conduction system, valves and atrial septum, the diversity of cardiac mesenchymal cells, autonomic innervation and chromaffin cells.

#### Libs: **src/libs**

- **single_cell_lib**

  - Class **SingleCell**: generic Scanpy, Squidpy, cell2location and LIANA methods

  - **DEFAULT_PARAMS**
- **single_cell_heart_dvlp_lib**
  - Class **SingleCellHeart(SingleCell)**: developing-heart methods (sections and ages, the paper's stereoscope proportions and spatial clusters, gene x age panels, regions vs the paper, temporal trends)

  - **HEART_PARAMS**
  - **HEART_LINEAGE**
    - 31 cell types in 9 lineages
  - **HEART_GENES**
    - 20 heart-development genes
  - **HEART_FILES**
   - one file registry per notebook: 'heart01' ... 'heart04'

<br>

| section | content |
|---|---|
| 1 | the complete Visium h5ad: 38 sections, 16 hearts, 9 ages; section table; H&E gallery |
| 2 | Visium quality control per section; MT genes |
| 3 | scRNA reference: components, QC per 10x library, cell types, composition by age |
| 4 | Figure 1: reference UMAP by cell type + dot plot of the 20 heart-development genes |
| 5 | the 20 genes on the reference UMAP by age group |
| 6 | the 20 genes on the Visium sections by age (one representative section per age) |
| 7 | cell2location: gene filter, NB regression of the reference signatures, training history, QC |
| 8 | cell2location: spatial mapping (cache of src/programs/run_c2l_heart_dvlp.py) |
| 9 | cell-type abundance on the sections by age (cell2location or the paper's stereoscope) |
| 10 | regions: Leiden on the abundances, UMAP + spatial, regions by age, interactive maps |
| 11 | NMF co-location (cell2location map only) |

**Data** (**heart_dvlp/data**, built by **src/programs/heart_dvlp_download.py**, **export_seurat_rds.R**, **heart_dvlp_build_h5ad.py**):

- **hdca_heart_visium_38sections.h5ad**: 69,115 spots x 36,601 genes, raw counts, hires + lowres H&E of every section; the paper's 23 spatial clusters (17 sections) and stereoscope proportions (33 coarse, 74 fine cell types) are attached.

- **hdca_heart_sc_reference.h5ad**: 76,991 cells x 35,239 genes, raw counts, the paper's annotation and UMAP (from **HDCA_heart_sc_full_extended.rds**).
- **heart_sections.tsv**: one row per section (age, heart, chambers, representative section of each age).

Figures and tables of this notebook have the prefix **heart01_**.

<div class="alert alert-warning">
<b>Differences from the paper</b>
</div>

- **No h5ad is published.** The two h5ad files were assembled here: Visium from the Space Ranger outputs (Mendeley part 1), the reference from the authors' Seurat object (Mendeley part 2). Spot QC: all 69,115 Space Ranger tissue spots are kept; 69,114 are in the paper's stereoscope table (**obs['in_paper_qc']**).

- **Sections per age.** The study has 38 sections (not one per week): w6 2, w7 5, w7.5 4, w8 7, w9 5, w10 4, w10.5 2, w11 5, w12 4. Figures by age show **one representative section per age** (most whole chambers, then most spots; **heart_sections.tsv**, column **representative**); tables use all sections (each section weighs the same).
- **Deconvolution.** The paper used **stereoscope**; this project uses **cell2location** (as the lymphoid and mouse-brain notebooks) with the same 31 high-level cell types. The full mapping (69,115 spots) needs the GPU: **python src/programs/run_c2l_heart_dvlp.py --mode all --tag gpu --accelerator gpu**, then **params['c2l_tag'] = 'gpu'**. Until the map exists, **abundance_source='auto'** uses the paper's stereoscope proportions (as **mp_mouse_brain01** used the paper's own cell2location map).
- **Regions** are Leiden clusters of the cell-type abundances (cell2location workflow); the paper's 23 spatial clusters are gene-expression clusters (BANKSY / Seurat) of 17 sections; both are compared in **mp_heart_dvlp02**.

## Loading packages

In [1]:
import os, sys
from pathlib import Path
from importlib.metadata import version
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import gc

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42          # editable text in PDFs
%config InlineBackend.print_figure_kwargs = {'bbox_inches': 'tight', 'dpi': 90}   # inline display only; files are saved at 300 dpi
import plotly.io as pio
pio.renderers.default = "notebook_connected"

ROOT0 = Path('/home/flavio/uv/cell_cell_communication')
ROOT_SRC = ROOT0 / 'src'
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.single_cell_heart_dvlp_lib import SingleCellHeart, HEART_PARAMS, HEART_LINEAGE, HEART_GENES, HEART_FILES
from libs.single_cell_lib import cmap_byr, cmap_lbwr
from libs.matplotlib_figure_style import apply_figure_style
from libs.Basic import create_dir

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 200)
print({p: version(p) for p in ['scanpy', 'squidpy', 'anndata', 'cell2location', 'scvi-tools', 'torch', 'liana']})

{'scanpy': '1.12.4', 'squidpy': '1.8.3', 'anndata': '0.13.4', 'cell2location': '0.1.5', 'scvi-tools': '1.5.1', 'torch': '2.14.1', 'liana': '2.1.1'}


## Parameters

All parameters and file names are in the class (**src/libs/single_cell_heart_dvlp_lib.py**):

- **HEART_PARAMS** (over **DEFAULT_PARAMS**) and **HEART_ANALYSIS_PARAMS['heart01']**; the figure style is **font_color**, **bg_color**, **fontsize_scale** (every plot method: argument None = these values);
- **HEART_FILES['inputs']** + **['heart01']**: key -> (root, name); **sch.fname(key)** gives the name for the methods (fname=...), **sch.file_path(key)** the full path. The cell2location folders carry **params['c2l_tag']** (default **gpu**: results/c2l_gpu).

Override in the constructor: `SingleCellHeart(ROOT0, analysis='heart01', params=dict(...), files=dict(...))`.

In [2]:
force=False
verbose=True

apply_figure_style()
# parameters: DEFAULT_PARAMS < HEART_PARAMS < HEART_ANALYSIS_PARAMS['heart01']  (params=dict(...) overrides)
# files:      HEART_FILES['inputs'] + HEART_FILES['heart01']                    (sch.fname(key) / sch.file_path(key))
sch = SingleCellHeart(ROOT0, analysis='heart01')
root_data, root_figure, root_result = sch.root_data, sch.root_figure, sch.root_result

ref_run_name = create_dir(sch.file_path('c2l_ref_dir'))     # reference_signatures: sc.h5ad + model.pt
run_name     = create_dir(sch.file_path('c2l_map_dir'))     # cell2location_map: sp.h5ad + model.pt

print(sch)
print("\n----------------\n")
print(root_data, sorted(os.listdir(root_data)))

SingleCellHeart(project='heart_dvlp', adata=None, symbol_col='symbol', sample_col='section')

----------------

/home/flavio/uv/cell_cell_communication/heart_dvlp/data ['.~lock.mendeley_files.tsv#', 'bundles', 'extended_tables', 'extended_text', 'hdca_heart_sc_reference.h5ad', 'hdca_heart_visium_38sections.h5ad', 'heart_regions.tsv', 'heart_sections.tsv', 'hires_bg', 'mendeley_files.tsv', 'metadata', 'rds', 'visium']


In [3]:
df_params = sch.calc_df_params()
print("parameters that differ from DEFAULT_PARAMS:")
display(df_params[df_params.value.astype(str) != df_params.default.astype(str)])
sch.calc_df_files()

parameters that differ from DEFAULT_PARAMS:


,parameter,value,default


,key,root,name,path,exists
0,visium,data,hdca_heart_visium_38sections.h5ad,/home/flavio/uv/cell_cell_communication/heart_dvlp/data/hdca_heart_visium_38sections.h5ad,True
1,reference,data,hdca_heart_sc_reference.h5ad,/home/flavio/uv/cell_cell_communication/heart_dvlp/data/hdca_heart_sc_reference.h5ad,True
2,sections,data,heart_sections.tsv,/home/flavio/uv/cell_cell_communication/heart_dvlp/data/heart_sections.tsv,True
3,visium_raw,data,visium,/home/flavio/uv/cell_cell_communication/heart_dvlp/data/visium,True
4,bundle_sc,data,bundles/sc_reference,/home/flavio/uv/cell_cell_communication/heart_dvlp/data/bundles/sc_reference,True
5,bundle_st,data,bundles/st_clustering,/home/flavio/uv/cell_cell_communication/heart_dvlp/data/bundles/st_clustering,True
6,st_annotations,data,metadata/HDCA_heart_ST_annotations.csv,/home/flavio/uv/cell_cell_communication/heart_dvlp/data/metadata/HDCA_heart_ST_annotations.csv,True
7,hl_annotations,data,metadata/HDCA_heart_SC_annotations_HL_240115.csv,/home/flavio/uv/cell_cell_communication/heart_dvlp/data/metadata/HDCA_heart_SC_annotations_HL_240115.csv,True
8,dl_annotations,data,metadata/HDCA_heart_SC_annotations_DL_240204.csv,/home/flavio/uv/cell_cell_communication/heart_dvlp/data/metadata/HDCA_heart_SC_annotations_DL_240204.csv,True
9,stereoscope_hl,data,metadata/W.2023-09-25133054.715121_hl.tsv,/home/flavio/uv/cell_cell_communication/heart_dvlp/data/metadata/W.2023-09-25133054.715121_hl.tsv,True


### 1. Visium data - adata_vis

The 38 Space Ranger outputs read with **sch.read_visium_samples** (obs_names = **<section>_<barcode>**, var_names = Ensembl IDs, symbols in **var['symbol']**; one **uns['spatial']** entry per section). **obs**: section metadata of **heart_sections.tsv** (age, heart, chambers), the paper's spatial cluster (**paper_cluster**, **paper_cluster_name**; 17 sections) and **in_paper_qc**. **obsm**: the paper's stereoscope proportions (**stereoscope_hl**: 33 coarse types; **stereoscope_dl**: 74 fine states).

csr: compressed sparse row:  is a storage format that saves memory and speeds up calculations by storing only the non-zero elements of a sparse matrix.

In [4]:
adata_vis = sch.read_h5ad('visium', set_self=False, verbose=verbose)
print(f"{os.path.getsize(sch.file_path('visium')) / 1e9:.2f} GB")
sch.summary(adata_vis)

Reading /home/flavio/uv/cell_cell_communication/heart_dvlp/data/hdca_heart_visium_38sections.h5ad
0.85 GB
AnnData object with n_obs × n_vars = 69115 × 36601
    obs: 'in_tissue', 'array_row', 'array_col', 'section', 'barcode', 'section_num', 'age_label', 'age_pcw', 'pcw', 'age_group', 'heart', 'chambers', 'n_chambers', 'in_paper_clustering', 'spot_id_paper', 'paper_cluster', 'paper_cluster_name', 'in_paper_qc'
    var: 'symbol'
    uns: 'dataset', 'spatial'
    obsm: 'spatial', 'stereoscope_dl', 'stereoscope_hl'
    layers: None (.X)

X: {'type': 'csr_matrix', 'dtype': 'float32', 'shape': (69115, 36601), 'density': 0.1513, 'min': 1.0, 'max': 6615.0, 'integer': True, 'nnz_per_row': 5537.7}

obs columns (18): ['in_tissue', 'array_row', 'array_col', 'section', 'barcode', 'section_num', 'age_label', 'age_pcw', 'pcw', 'age_group', 'heart', 'chambers', 'n_chambers', 'in_paper_clustering', 'spot_id_paper', 'paper_cluster', 'paper_cluster_name', 'in_paper_qc']
var columns (1): ['symbol']
obsm:

In [5]:
df_comp_vis = sch.calc_df_components(adata_vis)
slots = df_comp_vis.slot.unique()

for slot in slots:
    df2 = df_comp_vis[df_comp_vis.slot == slot]
    print(df2.head(6).to_string())
    print()

  slot key        type           shape
0    X      csr_matrix  (69115, 36601)

  slot          key      type     shape
1  obs    in_tissue     int64  (69115,)
2  obs    array_row     int64  (69115,)
3  obs    array_col     int64  (69115,)
4  obs      section  category  (69115,)
5  obs      barcode  category  (69115,)
6  obs  section_num     int64  (69115,)

   slot     key    type     shape
19  var  symbol  object  (36601,)

    slot             key       type        shape
20  obsm         spatial    ndarray   (69115, 2)
21  obsm  stereoscope_dl  DataFrame  (69115, 74)
22  obsm  stereoscope_hl  DataFrame  (69115, 33)

      slot   key        type           shape
23  layers  None  csr_matrix  (69115, 36601)

   slot      key  type                                                                                                   shape
24  uns  dataset  dict                                                           ['n_hearts', 'n_sections', 'paper', 'source']
25  uns  spatial  dict  ['V10

In [6]:
df_comp_vis.groupby('slot').size().to_frame('n_keys')

,n_keys
slot,
X,1
layers,1
obs,18
obsm,3
uns,2
var,1


In [7]:
df_comp_vis[df_comp_vis.slot == 'var']

,slot,key,type,shape
19,var,symbol,object,"(36601,)"


In [8]:
adata_vis.obs.head(3).T

,V10A13-160_A1_AAACGAGACGGTTGAT-1,V10A13-160_A1_AAACTGCTGGCTCCAA-1,V10A13-160_A1_AAAGGCTCTCGCGCCG-1
in_tissue,1,1,1
array_row,35,45,55
array_col,79,67,55
section,V10A13-160_A1,V10A13-160_A1,V10A13-160_A1
barcode,AAACGAGACGGTTGAT-1,AAACTGCTGGCTCCAA-1,AAAGGCTCTCGCGCCG-1
section_num,5,5,5
age_label,w6,w6,w6
age_pcw,6.0,6.0,6.0
pcw,6,6,6
age_group,w6-w7,w6-w7,w6-w7


In [9]:
dfo = adata_vis.obs

age_labels =np.unique(dfo.age_label)
len(age_labels), age_labels

(9,
 array(['w10', 'w10.5', 'w11', 'w12', 'w6', 'w7', 'w7.5', 'w8', 'w9'],
       dtype=object))

In [10]:
age_groups =np.unique(dfo.age_group)
len(age_groups), age_groups

(3, array(['w10-w12', 'w6-w7', 'w8-w9'], dtype=object))

In [11]:
chambers =np.unique(dfo.chambers)
len(chambers), chambers

(9,
 array(['LA,RA', 'LV,LA', 'LV,RV', 'LV,RV,LA', 'LV,RV,LA,RA', 'LV,RV,RA',
        'RV', 'RV,LA', 'RV,RA'], dtype=object))

#### Sections, hearts and ages

- **age_label** = postconceptional week of the heart (w7.5 and w10.5 are separate hearts)

- **age_group** = the authors' grouping (w6-w7, w8-w9, w10-w12; ST_export_data.Rmd).
- **chambers** = whole chambers in the section (LV, RV, LA, RA; partially present chambers in **chambers_partial**). 

The paper's 23 spatial clusters were computed on the 17 sections with at least three chambers (**in_paper_clustering**).

In [12]:
df_sections = sch.section_table(adata_vis)

print(np.array(df_sections.columns), '\n')

cols=['section_id', 'section_num', 'age_label', 'age_group', 'heart', 'chambers', 'chambers_partial',
      'n_chambers', 'n_spots', 'in_paper_clustering', 'representative', 'LV', 'RV', 'LA', 'RA', 'comment']
df_sections[cols].head(3).T

['section_id' 'section_num' 'age_label' 'age_pcw' 'pcw' 'age_group'
 'heart' 'LV' 'RV' 'LA' 'RA' 'chambers' 'chambers_partial' 'n_chambers'
 'comment' 'in_paper_clustering' 'n_spots' 'representative'
 'n_spots_adata'] 



,0,1,2
section_id,V10A13-160_A1,V10A13-160_B1,V10A13-160_C1
section_num,5,6,7
age_label,w6,w6,w7
age_group,w6-w7,w6-w7,w6-w7
heart,XDD363,XDD363,XDD398
chambers,"LV,RV,RA","LV,RV,RA","LV,RV,RA"
chambers_partial,,,
n_chambers,3,3,3
n_spots,615,575,746
in_paper_clustering,True,True,True


In [14]:
df_age_group = df_sections.groupby('age_group', sort=False).agg(n_sections=('section_id', 'size'), 
                                                          n_hearts=('heart', 'nunique'),
                                                          n_spots=('n_spots', 'sum'),
                                                          representative=('section_id', lambda s: ', '.join(s[df_sections.loc[s.index, 'representative']])))
df_age_group

,n_sections,n_hearts,n_spots,representative
age_group,,,,
w6-w7,11,5,11704,"V10A13-160_A1, V10A13-160_C1, V11A20-402_C1"
w8-w9,12,5,14254,"V11A20-401_A1, V10A13-157_C1"
w10-w12,15,6,43157,"V10F24-105_D1, V11A20-401_D1, V10T03-307_B1, V10T03-309_C1"


In [15]:
df_age = df_sections.groupby('age_label', sort=False).agg(n_sections=('section_id', 'size'), 
                                                          n_hearts=('heart', 'nunique'),
                                                          n_spots=('n_spots', 'sum'),
                                                          representative=('section_id', lambda s: ', '.join(s[df_sections.loc[s.index, 'representative']])))
df_age

,n_sections,n_hearts,n_spots,representative
age_label,,,,
w6,2,1,1190,V10A13-160_A1
w7,5,2,6774,V10A13-160_C1
w7.5,4,2,3740,V11A20-402_C1
w8,7,3,7532,V11A20-401_A1
w9,5,2,6722,V10A13-157_C1
w10,4,1,9107,V10F24-105_D1
w10.5,2,1,5304,V11A20-401_D1
w11,5,2,16166,V10T03-307_B1
w12,4,2,12580,V10T03-309_C1


In [ ]:
sections_rep = np.array(sch.representative_sections())
print(f"representative sections (one per age) ({len(sections_rep)}): {sections_rep}")

#### H&E of the 38 sections

Glass outside the tissue in black (**params['image_bg_color']**); title: age | heart, section | whole chambers | spots.

In [ ]:
force=False
verbose=True

fig = sch.plot_sections_gallery(adata_vis, ncols=5, panel_size=4, 
                                fname=sch.fname('fig_sections'), 
                                force=force, verbose=verbose)
plt.show()

#### One representative section per age

In [ ]:
sections_rep

In [ ]:
force=False
verbose=True

fig = sch.plot_sections_gallery(adata_vis, sections=sections_rep, 
                                ncols=5, panel_size=4,
                                fname=sch.fname('fig_sections_rep'),
                                force=force, verbose=verbose)
plt.show()

### 2. Quality control - Visium

QC metrics per spot (**sch.calc_qc_metrics**: genes, UMIs, % mitochondrial), violins per age; medians and quartiles per section in the table. Large sections of w11-w12 cover the whole capture area (the hearts are larger than the 6.5 mm frame), so they are partial views of the heart.

In [ ]:
sch.calc_qc_metrics(adata_vis)
force=False
verbose=True
fig = sch.plot_qc(groupby='age_label', adata=adata_vis, figsize=(22, 5), fontsize_scale=2.5, fname=sch.fname('fig_qc_visium'),
                  title='Visium spots per age (all sections of the age; per section: table below)', force=force, verbose=verbose)
plt.show()

In [ ]:
df_qc_vis = sch.calc_df_section_qc(adata=adata_vis, df_sections=df_sections, cols=['age_label', 'heart'], force=force, verbose=verbose)
df_qc_vis.head(5).T

#### Mitochondria-encoded (MT) genes

Removed for spatial mapping; their counts are kept in **obsm['MT']**.

In [ ]:
n0 = adata_vis.n_vars
adata_vis = sch.remove_mt_genes(adata_vis)
print(f"genes {n0:,} -> {adata_vis.n_vars:,}; obsm['MT'] {adata_vis.obsm['MT'].shape}")

### 3. The scRNA reference: hdca_heart_sc_reference.h5ad

The authors' Seurat object **HDCA_heart_sc_full_extended.rds** (Mendeley part 2) exported with **src/programs/export_seurat_rds.R** (SeuratObject only) and assembled by **sch.build_reference_h5ad**:

- **X**: raw UMI counts of 76,991 cells; genes renamed to the Ensembl IDs of the Visium features (35,239 genes shared with Visium; 22,905 features of the Cell Ranger reference without a unique Ensembl ID, incl. transgene markers, dropped; a median 98.4% of each cell's UMIs is kept);
- **obs['cell_type']**: the 31 high-level (coarse-grained) cell types + 4 **HL_excl_*** clusters the authors removed (red-blood-cell dominated / low quality; **obs['excluded']**); **cluster** = the high-level cluster id; **lineage** (**HEART_LINEAGE**);
- **sample_id**: 21 10x libraries from 15 hearts (**heart_id**); **age** (PCW 5.5-14) and **age_group** (5-6, 7-8, 9-11, 12-14 PCW); QC columns (**n_counts**, **n_genes**, **percent_mito**, doublet scores, cell-cycle phase);
- **obsm['X_umap']**: the paper's UMAP (**umap_oi**); also **X_umap_df**, **X_pca**, **X_harmony_oi**.

In [ ]:
adata_ref = sch.read_h5ad('reference', set_self=False, verbose=verbose)
print(f"{os.path.getsize(sch.file_path('reference')) / 1e9:.2f} GB")
sch.summary(adata_ref)

In [ ]:
print("X:", sch.x_info(adata_ref))
print("raw counts:", sch.is_raw_counts(adata_ref))
obs_cols = ['cell_type', 'lineage', 'excluded', 'sample_id', 'heart_id', 'age', 'age_group', 'Phase']
for c, vc in sch.value_counts(obs_cols, adata_ref).items():
    print(f"\n## {c} ({adata_ref.obs[c].nunique(dropna=False)} levels)")
    print(vc.to_string())

In [ ]:
cols = ['n_counts', 'n_genes', 'percent_mito', 'percent_ribo', 'percent_hb', 'DoubletScore']
adata_ref.obs[cols].describe().round(2)

#### QC per 10x library

**plot_qc** needs **n_genes_by_counts**, **total_counts**, **pct_counts_mt** (scanpy names).

In [ ]:
sch.calc_qc_metrics(adata_ref)
force=False
verbose=True

fig = sch.plot_qc(groupby='sample_id', adata=adata_ref, figsize=(30, 5), fontsize_scale=2, fname=sch.fname('fig_qc_ref'),
                  title='scRNA cells per 10x library', force=force, verbose=verbose)
plt.show()

#### The 31 cell types (the 4 HL_excl_* removed)

Abbreviations (paper): CM cardiomyocytes (v ventricular, a atrial; Mat mature, MetAct metabolically active, Immat immature, Prol proliferating); FB fibroblasts (Int interstitial, OFT outflow tract, AnnFibr annulus fibrosus, PDE4Chigh); EpC epicardial cells; EPDC epicardium-derived cells; MC mesenchymal cells (Valve, Peric pericardial); SMC smooth muscle cells (OFT, CA coronary artery); PC pericytes; EC endothelial cells (Endoc endocardial, EndocCush endocardial cushion, MicroVasc, MacroVasc, PDE4Chigh); LEC lymphatic EC; TMSB10high_C thymosin-high cells; NB-N neuroblasts-neurons; SCP-GC Schwann cell precursors - glial cells; MyC myeloid cells; LyC lymphoid cells.

In [ ]:
adata_ref = sch.filter_reference(adata_ref)
print(adata_ref.shape)

adata_ref.obs.head(3).T

In [ ]:
force=False
verbose=True

df_ct = sch.calc_df_reference_celltypes(adata_ref, lineage=HEART_LINEAGE, force=force, verbose=verbose)
df_ct

#### Cell-type composition by age (scRNA)

% of the cells of each age group. **Caveat**: dissociation and the number of hearts per age (1-5) change the composition; it is not a census of the tissue (the Visium abundances are, within the limits of deconvolution).

In [ ]:
# sch = SingleCellHeart(ROOT0, analysis='heart01')
import importlib

import libs.single_cell_heart_dvlp_lib as m   # the module where SingleCellHeart is defined
importlib.reload(m)
from libs.single_cell_heart_dvlp_lib import SingleCellHeart

sch = SingleCellHeart(ROOT0, analysis='heart01')

In [ ]:
force=False
verbose=False

# 1. counts: number of reference cells per cell type and age group (no normalization)
# 2. normalized by row (cell type): in which ages each cell type is frequent; every row sums to 100
#    (rescales the column percentages, so the unequal number of cells per age is corrected)
# 3. normalized by column (age group): composition of each age; every column sums to 100
df_ref_age_n = sch.calc_df_ref_age_composition(adata_ref, normalize='none', fname=sch.fname('tsv_ref_age_counts'), force=force, verbose=verbose)
df_ref_age_row = sch.calc_df_ref_age_composition(adata_ref, normalize='row', fname=sch.fname('tsv_ref_age_comp_row'), force=force, verbose=verbose)
df_ref_age = sch.calc_df_ref_age_composition(adata_ref, normalize='col', fname=sch.fname('tsv_ref_age_comp'), force=force, verbose=verbose)

for df_, norm, key in [(df_ref_age_n, 'none', 'fig_ref_age_counts'), (df_ref_age_row, 'row', 'fig_ref_age_comp_row'),
                       (df_ref_age, 'col', 'fig_ref_age_comp')]:
    fig = sch.plot_ref_age_composition(df_, normalize=norm, figsize=(12, 16), fname=sch.fname(key),
                                       fontsize_scale=2.6, cmap='Reds', force=force, verbose=verbose)
    plt.show()

### 4. Figure 1: reference UMAP by cell type + dot plot of 20 heart-development genes

#### The gene panel (**HEART_GENES**)

| group | genes | what they mark |
|---|---|---|
| core cardiac transcription factors | *NKX2-5*, *GATA4*, *TBX5*, *TBX20* | cardiomyocyte identity; *TBX5* left ventricle / atria, septation; *TBX20* chambers and valves |
| heart fields | *HAND1*, *HAND2*, *ISL1* | first heart field / left ventricle (*HAND1*); second heart field, right ventricle and outflow tract (*HAND2*, *ISL1*); *ISL1* also marks the sinoatrial node |
| chamber myosins / peptides | *MYH6*, *MYL7*, *NPPA* (atrial), *MYH7*, *MYL2*, *IRX4* (ventricular) | atrial vs ventricular myocardium; *NPPA* also trabecular ventricular myocardium |
| left-right and right atrium | *PITX2*, *BMP10* | left atrium (*PITX2*); right atrium and trabeculation (*BMP10*) |
| pacemaker-conduction system | *TBX3*, *SHOX2*, *HCN4* | sinoatrial node (*SHOX2*, *HCN4*, *TBX3*), atrioventricular node and bundle (*TBX3*) |
| epicardium | *WT1*, *TCF21* | epicardium and epicardium-derived fibroblast progenitors |

In [ ]:
genes_plot = list(HEART_GENES)
df_genes = sch.calc_df_gene_panel(adata_ref, adata_vis, genes=HEART_GENES, force=force, verbose=verbose)
df_genes

In [ ]:
force=False
verbose=True
df_dot = sch.plot_umap_celltype_dotplot(adata_ref, genes_plot, lineage=HEART_LINEAGE, figsize=(28, 18), 
                                        fontsize_scale=2.1, fontsize_x=10, fontsize_y=10,
                                        fname=sch.fname('fig_umap_dotplot'),
                                        force=force, verbose=verbose)
plt.show()

In [ ]:
# cell type with the highest mean expression of each gene (and its % expressing)
df_best = sch.calc_df_gene_best_celltype(df_dot, genes=genes_plot, lineage=HEART_LINEAGE, force=force, verbose=verbose)
print(df_best.shape)
df_best

### 5. The 20 genes on the reference UMAP by age group

Rows = genes, columns = scRNA age groups (PCW 5-6, 7-8, 9-11, 12-14); grey = all cells, colour = cells of the age group, log1p(CP10k) clipped at the 99.2th percentile, one scale per gene shared by the columns.

**Caveat**: the age groups have different hearts and cell-type compositions (section 3), so a change between columns can come from composition, donor or dissociation as well as from development.

In [ ]:
force=False
verbose=True
df_umap_age = sch.plot_umap_genes_by_age(genes_plot, adata_ref, panel_size=3.0, point_size=0.6, fontsize_scale=2.1,
                                         fname=sch.fname('fig_umap_genes_age'), force=force, verbose=verbose)
plt.show()

### **pct_expr**

**pct_expr** is the **percentage of scRNA reference cells in an age group with at least one UMI of the gene**. It's a column of the table returned by **plot_umap_genes_by_age**, the method that draws the heart-gene UMAP panels by age in notebook 01 (**df_umap_age**).

It is computed in **single_cell_heart_dvlp_lib.py**, line 1029:

```python
pct_expr = round(100 * float((v[m] > 0).mean()), 2)
```

- **v** is the gene's log1p(CP10k) expression in each reference cell. The library size is **obs['n_counts']**.
- **m** is the cells of one age group (PCW 5-6, 7-8, 9-11 or 12-14).
- **v > 0** is true for any cell with a count of 1 or more; log1p does not change which cells are zero.

The other columns of the same table are:
- **n_cells**: the number of cells in the age group.
- **mean_log1p_cp10k**: the mean log1p(CP10k) over **all** cells of the group, zeros included.

Notebook 01 pivots it into a gene × age table with **df_umap_age.pivot_table(index='gene', columns='age_group', values='pct_expr')**.

When reading it:
- **All cell types are pooled.** A change across ages can come from a shift in cell-type composition rather than a change inside a cell type. For example, MYH6 or NPPA rises when the share of atrial cardiomyocytes rises. The per-cell-type version is the dot plot (**% cells > 0** legend), and **heart01_gene_best_celltype.tsv** gives the cell type with the highest expression of each gene.
- **It measures detection, not level.** The 10x dropout rate makes it lower than the true fraction for weakly expressed genes, such as TBX5, HCN4 and SHOX2.

In [ ]:
df_umap_age.head(3)

In [ ]:
df_umap_age.pivot_table(index='gene', columns='age_group', values='pct_expr', sort=False)

### 6. The 20 genes on the Visium sections by age

One representative section per age (w6 ... w12); colour = log1p(CP10k) of the spot, one scale per gene shared by the sections (0 - 99.2th percentile). 5 genes per figure.

In [ ]:
force=False
verbose=False
figs = sch.plot_spatial_genes_by_age(genes_plot, adata_vis, sections=sections_rep, nframe=5, panel_size=2.6,
                                     fontsize_scale=2.0, fname=sch.fname('fig_genes_age'), force=force, verbose=verbose)
plt.show()

### 7. cell2location: gene filter and reference signatures (NB regression)

Permissive gene selection (**filter_genes_c2l**, cell2location defaults of **DEFAULT_PARAMS**: keeps markers of rare cell types). The regression model (**sch.fit_or_load_reference**): labels **cell_type** (31 types), batch **sample_id** (21 10x libraries), no extra covariate; signatures = posterior **q50** (**params['c2l_signature_summary']**).

**Cache**: the model is trained by **src/programs/run_c2l_heart_dvlp.py** (GPU, 250 epochs, tag **gpu**: python src/programs/run_c2l_heart_dvlp.py --mode all --tag gpu --accelerator gpu). The notebook trains only when the cache is missing and **params['c2l_train_if_missing']** is True.

In [ ]:
# !sed -i '1i import anndata' /home/flavio/uv/cell_cell_communication/.venv/lib/python3.12/site-packages/cell2location/utils/filtering.py

In [ ]:
n0 = adata_ref.n_vars
adata_ref_c2l, selected = sch.filter_genes_c2l(adata_ref, figsize=(5, 4))
plt.show()
print(f"genes {n0:,} -> {adata_ref_c2l.n_vars:,}")

In [ ]:
(ref_run_name / 'sc.h5ad').exists(), ref_run_name

In [ ]:
if (ref_run_name / 'sc.h5ad').exists() or sch.params['c2l_train_if_missing']:
    mod_reg, adata_ref_c2l, ref_trained = sch.fit_or_load_reference(
        adata_ref_c2l, ref_run_name, labels_key=sch.params['ref_labels_key'], batch_key=sch.params['ref_batch_key'],
        categorical_covariate_keys=list(sch.params['ref_covariate_keys']) or None)
    mod_reg.view_anndata_setup()
else:
    mod_reg = None
    print(f"no cached reference model in {ref_run_name}: python src/programs/run_c2l_heart_dvlp.py --mode reference --tag {sch.params['c2l_tag']}")

In [ ]:
force=False
verbose=True
if mod_reg is not None:
    fig = sch.plot_c2l_history(mod_reg, 10, fname=sch.fname('fig_ref_history'), force=force, verbose=verbose)
    plt.show()

**QC of the signatures** (cell2location): 1. reconstruction accuracy (2D histogram along a noisy diagonal); 2. estimated signatures vs the mean expression per cluster (differ because of batch effects).

In [ ]:
force=False
verbose=True
if mod_reg is not None:
    figs = sch.plot_c2l_qc(mod_reg, summary_name="q50", figsize=(5, 5), fname=sch.fname('fig_ref_qc'), force=force, verbose=verbose)
    df_inf_aver, summary_used = sch.get_reference_signatures(adata_ref_c2l, prefer=(sch.params['c2l_signature_summary'],))
    print(f"Using {summary_used} signatures:", df_inf_aver.shape)
    display(df_inf_aver.iloc[:5, :6])

### 8. cell2location: spatial mapping

**N_cells_per_location = 20** (**params['c2l_N_cells_per_location']**): the fetal myocardium is dense; estimate it from nuclei counts on the H&E for a final run. **detection_alpha = 20** (human tissue, technical variation between sections). Batch = section (38). The per-cell-type expected-expression layers are **not** computed (38 sections x 31 types would be about 100 GB); **calc_celltype_expr** (mp_heart_dvlp04) gives the same values when needed.

The mapping of 69,115 spots needs the GPU (> 25 min per full-data epoch on 20 CPU threads):

```bash
python src/programs/run_c2l_heart_dvlp.py --mode mapping --tag gpu --map_epochs 30000 --accelerator gpu   # + --batch_size 10000 if the GPU memory is short
```

In [ ]:
if (run_name / 'sp.h5ad').exists() or (sch.params['c2l_train_if_missing'] and mod_reg is not None):
    mod_c2l, adata_vis, map_trained = sch.fit_or_load_mapping(adata_vis, df_inf_aver, run_name)
    mod_c2l.view_anndata_setup()
else:
    mod_c2l = None
    print(f"no cached cell2location map in {run_name}:\n  python src/programs/run_c2l_heart_dvlp.py --mode mapping "
          f"--tag gpu --accelerator gpu  (then params c2l_tag='gpu')")
print("abundance source of this notebook:", sch.abundance_source(adata_vis))

In [ ]:
force=False
verbose=True

if mod_c2l is not None:
    fig = sch.plot_c2l_history(mod_c2l, 1000, label='full data training', fname=sch.fname('fig_map_history'), force=force, verbose=verbose)
    plt.show()
    figs = sch.plot_c2l_qc(mod_c2l, summary_name="q50", figsize=(5, 5), fname=sch.fname('fig_map_qc'), force=force, verbose=verbose)

### 9. Cell-type abundance on the sections by age

**sch.get_abundance**: cell2location q05 abundance (cells per spot, 'at least this many') when the map exists, otherwise the paper's **stereoscope** proportions (**obsm['stereoscope_hl']**, rows sum to 1; the HL_excl_* columns removed). With both, **calc_df_c2l_vs_stereoscope** gives the agreement per cell type.

In [ ]:
df_abund = sch.get_abundance(adata_vis)

print(df_abund.attrs['source'], df_abund.shape)
df_abund.describe().T.round(3).head(10)

In [ ]:
force=False
verbose=True

celltypes_8 = ['Mat_vCM', 'Mat_aCM', 'Immat_CM', 'Valve_MC', 'EndocCush_EC', 'EPDC', 'OFT_SMC', 'NB-N']
label = 'cells per spot (q05)' if df_abund.attrs['source'] == 'c2l' else 'stereoscope proportion'

fig = sch.plot_spatial_celltypes_by_age(celltypes_8, adata_vis, df_abund, sections=sections_rep, 
                                        panel_size=2.6, cbar_label=label, fontsize_scale=1.2, 
                                        fname=sch.fname('fig_abund_age'), force=force, verbose=verbose)
plt.show()

### 10. Regions: Leiden clustering of the spots on the cell-type abundances

**sch.cluster_regions**: kNN (**region_n_neighbors** = 30) on the abundances of the 31 types, Leiden (**region_resolution** = 1.0, igraph, seed 0) jointly over the 38 sections, so the regions are comparable between sections and ages. **sch.add_region_ids**: **region_label** R00 ... (= Leiden cluster) and an automatic **region_name** (top cell types); the curated names of **heart_regions.tsv** (written for the stereoscope regions; **abundance_source** column) are used when they match the source.

**Caveat**: the 38 sections are not batch corrected in abundance space; a region found in only one heart would be a technical or sectioning effect (see the n_sections column and mp_heart_dvlp02).

In [ ]:
force=True
verbose=True

sch.cluster_regions(adata_vis, df_abund=df_abund, force=force, verbose=verbose)

In [ ]:
force=True
verbose=True

sch.umap_regions(adata_vis, force=force, verbose=verbose)

In [ ]:
force=False
verbose=True

names_auto, df_region_info, names, df_cur = sch.calc_region_names(adata_vis, df_abund=df_abund,
                                                                   fname=sch.fname('tsv_region_names'),
                                                                   force=force, verbose=verbose)
print(df_cur.shape)
df_cur

In [ ]:
print(df_region_info.shape)
df_region_info.tail(3)

In [ ]:
len(names_auto), np.array(names_auto)

In [ ]:
len(names), np.array(names)

In [ ]:
force=False
verbose=True

df_regions = sch.save_region_labels(adata_vis, abundance_source=df_abund.attrs['source'], force=force, verbose=verbose)
print(df_regions.shape)
df_regions.head(3)

In [ ]:
force=False
verbose=True

fig = sch.umap_spatial_3plot(adata_vis, names=names, key='region_label',
                             library_id=sch.params['library_main'], figsize=(22, 9),
                             fname=sch.fname('fig_regions'), force=force, verbose=verbose)
plt.show()

In [ ]:
force=False
verbose=True

# interactive version (HTML in heart_dvlp/figures): one legend drives the region UMAP and the section map
fig = sch.plotly_umap_spatial_3plot(adata_vis, names=names, key='region_label', library_id=sch.params['library_main'],
                                    width=1500, height=720, fontsize_legend=14,
                                    fname='fig_regions_3plot_html', force=force, verbose=verbose)
fig.show()

#### Regions on one section per age

In [ ]:
force=False
verbose=False

fig = sch.plot_regions_by_age(adata_vis, key='region_label', sections=sections_rep, 
                              names=names, ncols=5, panel_size=4,
                              legend_ncol=3, fname=sch.fname('fig_regions_age'), 
                              bottom=0.3, wspace=0.02, hspace=0.12,
                              force=force, verbose=verbose)
plt.show()

In [ ]:
force=False
verbose=True

# interactive version (HTML in heart_dvlp/figures): click a region to hide it, double-click to isolate it in every section
fig = sch.plotly_regions_by_age(adata_vis, key='region_label', sections=sections_rep, 
                                names=names, ncols=3,
                                fname='fig_regions_age_html', force=force, verbose=verbose)
fig.show()

#### With Plotly, interactive (also saved as HTML), one file per representative section

In [ ]:
force=False
verbose=False

for lib in [sch.params['library_main']]:
    fig = sch.plotly_spatial_visium(names, adata_vis, key='region_label', library_id=lib, fontsize_title=20, fontsize_legend=13,
                                    fontsize_hover=13, marker_line_color="#222222", fname='fig_regions_html', force=force, verbose=verbose)
    fig.show()

In [ ]:
force=False
verbose=False

# the other representative sections: HTML files only (open them from heart_dvlp/figures)
for lib in sections_rep:
    if lib != sch.params['library_main']:
        fig = sch.plotly_spatial_visium(names, adata_vis, key='region_label', 
                                        library_id=lib, marker_line_color="#222222", 
                                        width=1200, height=750,
                                        fname='fig_regions_html', force=force, verbose=verbose)

        fig.show()

### 11. NMF co-location

cell2location **run_colocation** (CoLocatedGroupsSklearnNMF) factorises the cell2location abundances into co-located cell-type groups (tissue compartments). It needs the cell2location map (**adata.uns['mod']**); with the stereoscope proportions the section is skipped.

In [ ]:
sch.params['coloc_n_facts']

In [ ]:
force=False
verbose=False

if df_abund.attrs['source'] == 'c2l':
    adata_vis, res_dict, coloc_edited = sch.run_colocation_cached(adata_vis, run_name)
    for n in sch.params['coloc_n_facts']:
        fig, df_load = sch.plot_factor_dotplot(adata_vis, n, lineage=HEART_LINEAGE, fname=sch.fname('fig_factor_dotplot', n=n),
                                               fname_tsv=sch.fname('tsv_factor_load', n=n), force=force, verbose=verbose)
        plt.show()
else:
    print("NMF co-location needs the cell2location map: skipped (stereoscope abundances)")

### Notes for interpretation

- **Ages are hearts.** Each age has 1-3 hearts; within-age variation (sections of one heart, hearts of the same age) is the yardstick for any temporal change.
- **Large late hearts.** From w11 the sections cover only part of the heart (the capture area is 6.5 mm); region shares of w11-w12 describe the part that was sectioned (e.g. no LV in the w11 sections of XDD388).
- **Abundance source.** The regions of this run come from the paper's stereoscope proportions; after the GPU cell2location run (tag gpu) the regions are recomputed and the curated names of **heart_regions.tsv** must be redone for that source.